In [1]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

In [2]:
def plot_zmp_ref_y(ns, dt, zmp_ref):
    plt.figure(figsize=(8, 4))
    plt.plot(np.arange(ns) * dt, zmp_ref[1,:], label='$z_y$ reference') 
    plt.xlabel('Time [s]')
    plt.ylabel('Position [m]')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [3]:
def plot_trajopt(ns, dt, r, u, zmp_ref):
    plt.figure(figsize=(8, 4))
    plt.plot(np.arange(ns + 1) * dt, r[1, :], label='$r_y$ (CoM position)')
    plt.plot(np.arange(ns) * dt, u[1,:], label='$z_y$ (ZMP position)')
    plt.plot(np.arange(ns) * dt, zmp_ref[1,:], label='$z_y$ reference')
    plt.xlabel('Time [s]')
    plt.ylabel('Position [m]')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [6]:
class MPCPlotter:

    def __init__(
        self,
        dt,
        horizon,
        y_limits=(-0.2, 0.2),
    ):
        self.dt = dt
        self.horizon = horizon
        self.y_limits = y_limits

        self.i = 0
        self.handle = None

    def update(self, t, x, u):

        # ---------------------------------------------------------
        # Current simulation time
        # ---------------------------------------------------------

        sim_t = self.i * self.dt

        # Current position in the circular buffer
        current_idx = self.i % self.horizon

        # ---------------------------------------------------------
        # Reorder circular buffers chronologically
        # ---------------------------------------------------------

        # The element after current_idx is the oldest element
        start = (current_idx + 1) % self.horizon

        x_plot = np.concatenate(
            (x[:, start:], x[:, :start]),
            axis=1
        )

        u_plot = np.concatenate(
            (u[:, start:], u[:, :start]),
            axis=1
        )

        # ---------------------------------------------------------
        # Time axis
        # ---------------------------------------------------------

        N = min(
            self.horizon,
            x_plot.shape[1],
            u_plot.shape[1]
        )

        time = sim_t + np.arange(N) * self.dt

        # ---------------------------------------------------------
        # Plot
        # ---------------------------------------------------------

        fig, ax = plt.subplots(figsize=(9, 4))

        ax.plot(
            time[:N],
            x_plot[1, :N],
            linewidth=2,
            label='$r_y$ (CoM position)'
        )

        ax.plot(
            time[:N],
            u_plot[1, :N],
            linewidth=2,
            label='$z_y$ (ZMP position)'
        )

        # Current time
        ax.axvline(
            sim_t,
            linestyle=':',
            linewidth=1.5
        )

        # ---------------------------------------------------------
        # SCROLLING WINDOW
        # ---------------------------------------------------------

        ax.set_xlim(
            sim_t,
            sim_t + (N - 1) * self.dt
        )

        ax.set_ylim(*self.y_limits)

        ax.set_xlabel('Time [s]')
        ax.set_ylabel('Position [m]')

        ax.grid(True)
        ax.legend()

        fig.tight_layout()

        # ---------------------------------------------------------
        # Update the existing Jupyter output
        # ---------------------------------------------------------

        if self.handle is None:
            self.handle = display(
                fig,
                display_id=True
            )
        else:
            self.handle.update(fig)

        plt.close(fig)

        # Next MPC iteration
        self.i += 1